<a href="https://colab.research.google.com/github/MozahidulAlam/Files/blob/main/Detaset_prepare.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
import pandas as pd
import numpy as np

In [11]:
# 1. Load dataset
df = pd.read_csv("/content/drive/MyDrive/DataAnalysis_course/student_data_raw_preprocessing.csv")

print("Original Shape:", df.shape)
print("\nMissing Values:")
print(df.isnull().sum())



Original Shape: (15, 8)

Missing Values:
ï»¿ Student ID    0
Student Name      0
Department        0
Attendance (%)    1
Quiz Marks        1
Midterm Marks     1
Final Marks       1
Semester          0
dtype: int64


In [36]:
df

,ï»¿_student_id,student_name,department,attendance_,quiz_marks,midterm_marks,final_marks,semester,total_marks
0,101,Ayesha,CSE,92.0,18.0,25.0,42.0,1,85.0
1,102,Rahim,CSE,78.0,14.0,19.0,34.0,1,67.0
2,103,Nusrat,EEE,85.0,16.0,22.0,38.0,1,76.0
3,104,Tanvir,CSE,67.0,11.0,15.0,28.0,2,54.0
4,105,Sadia,BBA,95.0,19.0,27.0,45.0,2,91.0
5,106,Imran,EEE,80.5,13.0,18.0,32.0,2,63.0
6,107,Mim,CSE,88.0,14.0,24.0,40.0,1,78.0
7,108,Hasan,BBA,61.0,10.0,21.0,25.0,3,56.0
8,109,Jannat,EEE,90.0,18.0,26.0,34.0,3,78.0
9,110,Rafi,CSE,83.0,15.0,21.0,36.0,2,72.0


In [20]:
# 2. Clean column names
df.columns = (
    df.columns.str.strip()
    .str.lower()
    .str.replace(r"\s+", "_", regex=True)
    .str.replace(r"[()%]", "", regex=True)
)

print("\nClean Column Names:")
print(df.columns.tolist())


Clean Column Names:
['ï»¿_student_id', 'student_name', 'department', 'attendance_', 'quiz_marks', 'midterm_marks', 'final_marks', 'semester']


In [13]:
# 3. Remove duplicate rows
print("\nDuplicate Rows:", df.duplicated().sum())
df = df.drop_duplicates()


Duplicate Rows: 1


In [14]:
# 4. Clean text columns
df["student_name"] = (
    df["student_name"].astype("string")
    .str.strip()
    .str.title()
)

df["department"] = (
    df["department"].astype("string")
    .str.strip()
    .str.upper()
)

In [32]:
df.head()

,ï»¿_student_id,student_name,department,attendance_,quiz_marks,midterm_marks,final_marks,semester,total_marks
0,101,Ayesha,CSE,92.0,18.0,25.0,42.0,1,85.0
1,102,Rahim,CSE,78.0,14.0,19.0,34.0,1,67.0
2,103,Nusrat,EEE,85.0,16.0,22.0,38.0,1,76.0
3,104,Tanvir,CSE,67.0,11.0,15.0,28.0,2,54.0
4,105,Sadia,BBA,95.0,19.0,27.0,45.0,2,91.0


In [21]:
# 5. Convert columns to numeric
numeric_cols = [
    "ï»¿_student_id", "attendance_",
    "quiz_marks", "midterm_marks",
    "final_marks", "semester"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

In [33]:
df.head()

,ï»¿_student_id,student_name,department,attendance_,quiz_marks,midterm_marks,final_marks,semester,total_marks
0,101,Ayesha,CSE,92.0,18.0,25.0,42.0,1,85.0
1,102,Rahim,CSE,78.0,14.0,19.0,34.0,1,67.0
2,103,Nusrat,EEE,85.0,16.0,22.0,38.0,1,76.0
3,104,Tanvir,CSE,67.0,11.0,15.0,28.0,2,54.0
4,105,Sadia,BBA,95.0,19.0,27.0,45.0,2,91.0


In [23]:
# 6. Replace invalid values with NaN
df.loc[
    ~df["attendance_"].between(0, 100),
    "attendance_"
] = np.nan

# Marks ranges:
# Quiz: 0-20, Midterm: 0-30, Final: 0-50
mark_limits = {
    "quiz_marks": 20,
    "midterm_marks": 30,
    "final_marks": 50
}

for col, maximum in mark_limits.items():
    df.loc[~df[col].between(0, maximum), col] = np.nan

In [24]:
# 7. Handle missing values using median
for col in [
    "attendance_", "quiz_marks",
    "midterm_marks", "final_marks"
]:
    df[col] = df[col].fillna(df[col].median())

In [26]:
# 8. Detect outliers using IQR
for col in [
    "attendance_", "quiz_marks",
    "midterm_marks", "final_marks"
]:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers = df[
        (df[col] < lower) | (df[col] > upper)
    ]

    print(f"\nOutliers in {col}:")
    print(outliers[["ï»¿_student_id", col]])


Outliers in attendance_:
Empty DataFrame
Columns: [ï»¿_student_id, attendance_]
Index: []

Outliers in quiz_marks:
Empty DataFrame
Columns: [ï»¿_student_id, quiz_marks]
Index: []

Outliers in midterm_marks:
Empty DataFrame
Columns: [ï»¿_student_id, midterm_marks]
Index: []

Outliers in final_marks:
Empty DataFrame
Columns: [ï»¿_student_id, final_marks]
Index: []


In [27]:
# 9. Create total marks
df["total_marks"] = (
    df["quiz_marks"]
    + df["midterm_marks"]
    + df["final_marks"]
)

In [35]:
df.head()

,ï»¿_student_id,student_name,department,attendance_,quiz_marks,midterm_marks,final_marks,semester,total_marks
0,101,Ayesha,CSE,92.0,18.0,25.0,42.0,1,85.0
1,102,Rahim,CSE,78.0,14.0,19.0,34.0,1,67.0
2,103,Nusrat,EEE,85.0,16.0,22.0,38.0,1,76.0
3,104,Tanvir,CSE,67.0,11.0,15.0,28.0,2,54.0
4,105,Sadia,BBA,95.0,19.0,27.0,45.0,2,91.0


In [28]:
# 10. Check final dataset
print("\nCleaned Dataset:")
print(df)

print("\nRemaining Missing Values:")
print(df.isnull().sum())

print("\nFinal Shape:", df.shape)


Cleaned Dataset:
    ï»¿_student_id student_name department  attendance_  quiz_marks  \
0              101       Ayesha        CSE         92.0        18.0   
1              102        Rahim        CSE         78.0        14.0   
2              103       Nusrat        EEE         85.0        16.0   
3              104       Tanvir        CSE         67.0        11.0   
4              105        Sadia        BBA         95.0        19.0   
5              106        Imran        EEE         80.5        13.0   
6              107          Mim        CSE         88.0        14.0   
7              108        Hasan        BBA         61.0        10.0   
8              109       Jannat        EEE         90.0        18.0   
9              110         Rafi        CSE         83.0        15.0   
11             112      Sumaiya        BBA         76.0        14.0   
12             113        Fahim        EEE         58.0         9.0   
13             114       Nabila        CSE         80.5    

In [29]:
# 11. Save cleaned dataset
df.to_csv("student_data_cleaned.csv", index=False)

print("\nCleaned dataset saved successfully!")


Cleaned dataset saved successfully!
